# Определение ботов по потоку событий

**Итоговая модель:** CatBoost с 42 признаками: 30 поведенческих и 12 совместных XY.
**Результат запуска:** `submission.csv`, совпадающий с сохраненным вариантом
`catboost_behavior_xy.csv`.

Для воспроизведения нужны этот ноутбук, зависимости и три исходных файла в `data/`.
Функции расчета признаков и метрики находятся прямо в ноутбуке. Готовая модель,
прежний submission и результаты экспериментов при запуске не читаются.

Запуск: выбрать Python 3.14.3 с зависимостями из `requirements-notebook.txt`,
затем **Restart Kernel and Run All**. Либо из папки проекта выполнить
`python run_solution.py`. Все вычисления и обучение выполняются локально.

## Подход к решению

Единица классификации — cookie с заданным суточным окном. Я преобразую события
каждой cookie в одну строку признаков и обучаю CatBoost предсказывать вероятность
`target=1`: принадлежности к известным источникам автоматизированного сбора данных.

Основная гипотеза: кроме объема действий важны **ритм, разнообразие и повторяемость**.
Медиана и разброс пауз описывают темп; доли уникальных значений и энтропия — концентрацию
на объявлениях, категориях, городах и запросах. Совместные XY-признаки описывают форму
облака позиций и смещения между наблюдаемыми точками.

До агрегации я удаляю полные дубликаты, ограничиваю историю окном `[start, end)`
и сортирую события внутри cookie. Пропуск не считаю отдельным объявлением,
запросом или координатой 0. Неопределенные статистики оставляю как NaN.
Преобразования не используют target и не объединяют поведение разных cookie.

Группы признаков сравнивались на временных разбиениях: обучение до 12 апреля →
проверка 12–14 апреля; обучение до 15 апреля → проверка 15–16 апреля.
Исходные 30 признаков выбраны из 80 кандидатов. Затем отдельно проверены 12 вариантов
добавления XY. Набор и параметры здесь **зафиксированы**, повторного подбора нет.

Сырые ID, абсолютные даты, координаты, версии ПО и строка user_agent в модель
не передаются. Разобранный UA, глубина выдачи и сессии исследовались, но не вошли
в исходные 30. Из 92 рассмотренных кандидатов используются только 42 выбранных.

CatBoost позволяет совместно учитывать нелинейные сочетания признаков и пропуски.
Из-за дисбаланса классов используется `auto_class_weights='Balanced'`. Главная
метрика — максимальный Precision среди порогов с Recall ≥ 0.70. Для отправки
сохраняются вероятности, а не выбранный порог и не бинарные ответы.

## 1. Окружение и фиксированная конфигурация

Версии библиотек влияют на численные результаты, поэтому проверяются явно.
Seed CatBoost равен 0, число потоков равно 4, обучение выполняется на CPU.
Порядок признаков также фиксируется: он должен совпадать при обучении и прогнозе.

In [1]:
import hashlib
import platform
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import auc, precision_recall_curve, roc_auc_score

# Эти версии использованы при создании исходного XY-submission.
EXPECTED_VERSIONS = {
    'numpy': '2.4.2',
    'pandas': '3.0.5',
    'catboost': '1.2.10',
    'scikit-learn': '1.8.0',
}
actual_versions = {name: version(name) for name in EXPECTED_VERSIONS}
if platform.python_version() != '3.14.3' or actual_versions != EXPECTED_VERSIONS:
    raise RuntimeError('Для точного воспроизведения используйте Python 3.14.3 и requirements-notebook.txt.')
print('Python:', platform.python_version())
display(pd.Series(actual_versions, name='version').to_frame())

# Поддерживаются запуск из папки ноутбука и запуск из корня репозитория.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'data' / 'train.csv').exists():
    PROJECT_DIR = PROJECT_DIR / 'bot_detection_challenge'
DATA_DIR = PROJECT_DIR / 'data'
if not (DATA_DIR / 'train.csv').exists():
    raise FileNotFoundError('Рядом с ноутбуком должна находиться папка data с исходными файлами.')

Python: 3.14.3


,version
numpy,2.4.2
pandas,3.0.5
catboost,1.2.10
scikit-learn,1.8.0


In [2]:
# Параметры и порядок столбцов перенесены из зафиксированной конфигурации XY.
MODEL_PARAMS = {'iterations': 500,
 'depth': 6,
 'learning_rate': 0.05,
 'l2_leaf_reg': 5,
 'loss_function': 'Logloss',
 'auto_class_weights': 'Balanced',
 'random_seed': 0,
 'thread_count': 4,
 'verbose': False,
 'allow_writing_files': False}

FEATURE_COLUMNS = [
    "gap_median",
    "gap_min",
    "gap_iqr_over_median",
    "item_unique_share",
    "category_entropy",
    "location_unique_share",
    "gap_le_10s_share",
    "location_top_share",
    "gap_unique_share",
    "query_top_share",
    "gap_cv",
    "category_top_share",
    "location_entropy",
    "gap_gt_30m_share",
    "event_entropy",
    "item_entropy",
    "query_unique_share",
    "item_top_share",
    "item_location_nunique",
    "n_item_view",
    "gap_max",
    "category_unique_share",
    "event_top_share",
    "event_unique_share",
    "gap_mean",
    "n_seller_page_view",
    "gap_std",
    "gap_mode_share",
    "item_category_nunique",
    "n_photo_swipe",
    "xy_n_points",
    "xy_observed_share",
    "xy_repeat_share",
    "xy_top_point_share",
    "xy_abs_correlation",
    "xy_anisotropy",
    "xy_radius_cv",
    "xy_zero_step_share",
    "xy_step_median_normalized",
    "xy_step_cv",
    "xy_turn_cos_mean",
    "xy_repeated_vector_share"
]

def make_model():
    """Каждый вызов создает новую модель с теми же настройками."""
    return CatBoostClassifier(**MODEL_PARAMS, cat_features=[])

print('Число признаков:', len(FEATURE_COLUMNS))

Число признаков: 42


## 2. Исходные данные и проверки

Контрольные суммы проверяют, что используется та же версия данных. Это не способ
генерации ответов: все score ниже вычисляются моделью, обученной с нуля.
`target` остается отдельно от признаков. Метаданные train и test можно объединить
для единого расчета, поскольку каждая агрегация ограничена одной cookie.

In [3]:
INPUT_SHA256 = {
    'train.csv': 'b659a0e9e3df8146a1c10ca33cace9ece3da016e0ce3a502cbabf6554b88c55f',
    'test.csv': '1a0342a55a30ff5bb0ee14428b91d2e18c759f55bf1ed35554b80f423f35f4b7',
    'events.csv.gz': 'fa6923a91846aed3f8212cde039732fcf8d818b1f9f44a57878bb133de185901',
}
for filename, expected_hash in INPUT_SHA256.items():
    if hashlib.sha256((DATA_DIR / filename).read_bytes()).hexdigest() != expected_hash:
        raise ValueError(f'Изменилось содержимое {filename}; эталонный результат относится к другой версии данных.')

date_columns = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=date_columns)
test = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=date_columns)
raw_events = pd.read_csv(DATA_DIR / 'events.csv.gz', parse_dates=['event_ts'])

assert train.cookie_id.is_unique and test.cookie_id.is_unique
assert set(train.cookie_id).isdisjoint(test.cookie_id)
assert train.target.isin([0, 1]).all()
meta = pd.concat([train.drop(columns='target'), test], ignore_index=True)
assert meta.window_start_ts.lt(meta.window_end_ts).all()
y = train.target.to_numpy()

print('Train:', train.shape, 'test:', test.shape, 'events:', raw_events.shape)
print(f'Доля ботов в train: {train.target.mean():.4f}')
display(raw_events.isna().mean().rename('missing_share').to_frame().round(3))

Train: (11091, 5) test: (4909, 4) events: (328905, 14)
Доля ботов в train: 0.0811


,missing_share
cookie_id,0.000
event_ts,0.000
eid,0.000
event_name,0.000
platform,0.000
user_agent,0.000
item_id,0.348
item_category,0.100
item_location,0.072
seller_type,0.407


## 3. Очистка и границы наблюдения

Полный дубликат удаляется **до** нормализации платформы. Разные реальные события
в одну секунду сохраняются. Окно полуоткрытое: событие ровно в `window_end_ts`
уже недоступно модели. Порядок строк файла нельзя использовать как порядок действий.
Названия платформ нормализуются только для корректного отбора web-координат.

In [4]:
def prepare_events(raw_events, meta):
    """Очистить события, ограничить окнами и упорядочить внутри cookie."""
    if not meta.cookie_id.is_unique:
        raise ValueError('Ожидается ровно одно окно на cookie.')

    events = raw_events.drop_duplicates().copy()
    windows = meta[['cookie_id', 'window_start_ts', 'window_end_ts']]
    events = events.merge(windows, on='cookie_id', validate='many_to_one')
    inside = events.event_ts.ge(events.window_start_ts) & events.event_ts.lt(events.window_end_ts)
    events = events.loc[inside].copy()

    events['platform'] = (
        events.platform.str.strip().str.lower()
        .replace({'iphone': 'ios', 'desktop': 'web'})
    )
    return events.sort_values(['cookie_id', 'event_ts'], kind='stable').reset_index(drop=True)


events = prepare_events(raw_events, meta)
print('Удалено полных дубликатов:', int(raw_events.duplicated().sum()))
print('Событий вне окон:', len(raw_events.drop_duplicates()) - len(events))
print('Событий для расчета признаков:', len(events))
assert events.event_ts.ge(events.window_start_ts).all()
assert events.event_ts.lt(events.window_end_ts).all()

Удалено полных дубликатов: 4863
Событий вне окон: 40209
Событий для расчета признаков: 283833


## 4. Пять счетчиков

В финальном наборе остались число категорий и городов, просмотры объявлений,
страницы продавцов и листания фотографий. Они дают контекст для временных
и относительных признаков. Для cookie без событий счетчики равны нулю.

In [5]:
def count_features(events, index):
    """Число известных категорий/городов и три выбранных типа действий."""
    counts = events.groupby('cookie_id', sort=False).agg(
        item_category_nunique=('item_category', 'nunique'),
        item_location_nunique=('item_location', 'nunique'),
    )
    action_types = ['item_view', 'seller_page_view', 'photo_swipe']
    actions = (
        pd.crosstab(events.cookie_id, events.event_name)
        .reindex(columns=action_types, fill_value=0)
        .add_prefix('n_')
    )
    return counts.join(actions).reindex(index).fillna(0)

## 5. Одиннадцать характеристик ритма

Интервал — разность соседних временных меток внутри cookie в секундах.
Медиана описывает обычный темп, минимум — самые быстрые действия, максимум —
перерывы. Среднее, стандартное отклонение и `std/mean` учитывают изменчивость.

`IQR/(median+1)` сравнивает разброс центральных 50% интервалов с типичным темпом;
прибавление одной секунды позволяет учитывать нулевые медианы. Доли пауз ≤10 секунд
и >30 минут описывают быстрые серии и длинные перерывы. Доля уникальных интервалов
и доля самого частого интервала характеризуют повторяемость темпа.

Первое событие не имеет предыдущего и не входит в знаменатели долей. Нулевые
интервалы между разными одновременными событиями сохраняются. Эти признаки
не предполагают, что все боты действуют идеально равномерно.

In [6]:
def rhythm_features(events, index):
    """Агрегировать интервалы, не смешивая разные cookie."""
    timed = events[['cookie_id', 'event_ts']].copy()
    timed['gap'] = timed.groupby('cookie_id', sort=False).event_ts.diff().dt.total_seconds()
    gaps = timed.groupby('cookie_id').gap
    features = gaps.agg(['median', 'mean', 'std', 'min', 'max']).add_prefix('gap_')

    # Нулевая средняя пауза делает std/mean неопределенным, а не равным нулю.
    features['gap_cv'] = features.gap_std / features.gap_mean.replace(0, np.nan)
    quartiles = gaps.quantile([0.25, 0.75]).unstack().reindex(columns=[0.25, 0.75])
    features['gap_iqr_over_median'] = (quartiles[0.75] - quartiles[0.25]) / (features.gap_median + 1)

    for name, condition in {
        'gap_le_10s_share': timed.gap.le(10),
        'gap_gt_30m_share': timed.gap.gt(1800),
    }.items():
        features[name] = condition.where(timed.gap.notna()).groupby(timed.cookie_id).mean()

    # Повторяемость считается по числу интервалов, а не числу событий.
    frequencies = timed.dropna(subset=['gap']).groupby(['cookie_id', 'gap']).size()
    features['gap_mode_share'] = frequencies.groupby(level=0).max() / gaps.count()
    features['gap_unique_share'] = gaps.nunique() / gaps.count().replace(0, np.nan)
    return features.reindex(index)

## 6. Четырнадцать признаков разнообразия

Для объявлений, категорий, городов, запросов и типов действий считаются
`unique_share` — число разных значений / число известных значений,
`top_share` — доля наиболее частого значения и энтропия `-sum(p * log(p))`.
Энтропия запросов не вошла в отобранный набор и здесь не вычисляется.

Например, доля уникальных объявлений делится на число событий с известным
`item_id`, а не на все события cookie. Поэтому отсутствие ID у поискового события
не превращается в повторный просмотр фиктивного объявления. Названия и сырые
идентификаторы в модель не попадают — используются только характеристики частот.

In [7]:
def diversity_features(events, index):
    """Описать концентрацию просмотров без использования конкретных ID и названий."""
    features = pd.DataFrame(index=index)
    sources = [
        ('item_id', 'item'), ('item_category', 'category'),
        ('item_location', 'location'), ('search_query', 'query'), ('event_name', 'event'),
    ]
    for column, prefix in sources:
        # groupby исключает пропуски; знаменатель содержит только известные значения.
        counts = events.groupby(['cookie_id', column]).size()
        known_count = counts.groupby(level=0).sum()
        probabilities = counts / counts.groupby(level=0).transform('sum')
        features[f'{prefix}_unique_share'] = counts.groupby(level=0).size() / known_count
        features[f'{prefix}_top_share'] = probabilities.groupby(level=0).max()
        if prefix != 'query':
            features[f'{prefix}_entropy'] = (
                -probabilities * np.log(probabilities)
            ).groupby(level=0).sum()
    return features

## 7. Совместная геометрия X/Y и доступность координат

Здесь используются только web-события с двумя конечными координатами.
Число точек и доля событий с XY помогают оценить доступность информации.
Повторяемость определяется по паре `(x,y)`, а не по каждому столбцу отдельно.

После центрирования вычисляются ковариация и дисперсии. Модуль корреляции
характеризует связь осей. Вытянутость облака равна
`(lambda_max - lambda_min) / (lambda_max + lambda_min)`: 1 для линии и 0 для
одинакового разброса во всех направлениях. `radius_cv` — отношение std/mean
расстояний до центра. Форма требует хотя бы трех точек, повторяемость — двух.

Перенос и одинаковое масштабирование обеих осей не меняют эти характеристики.
Размер экрана неизвестен; разный масштаб осей и верстка все еще могут влиять
на форму. Геометрия не является доказательством конкретного механизма работы бота.

In [8]:
def pointer_cloud_features(events, index):
    """Семь признаков наличия XY, повторяемости пар и геометрии облака."""
    web = events.platform.eq('web')
    paired = web & np.isfinite(events[['pointer_x', 'pointer_y']]).all(axis=1)
    points = events.loc[paired].copy()
    n_points = points.groupby('cookie_id').size().reindex(index, fill_value=0)
    n_web = web.groupby(events.cookie_id).sum().reindex(index, fill_value=0)
    features = pd.DataFrame(index=index)
    features['xy_n_points'] = n_points.astype(float)
    features['xy_observed_share'] = n_points / n_web.replace(0, np.nan)

    # Совпадение обеих координат означает повтор наблюдаемой позиции.
    counts = points.groupby(['cookie_id', 'pointer_x', 'pointer_y']).size()
    features['xy_repeat_share'] = (1 - counts.groupby(level=0).size() / n_points).where(n_points.ge(2))
    features['xy_top_point_share'] = (counts.groupby(level=0).max() / n_points).where(n_points.ge(2))

    # Центрирование удаляет зависимость от положения облака на странице.
    points['cx'] = points.pointer_x - points.groupby('cookie_id').pointer_x.transform('mean')
    points['cy'] = points.pointer_y - points.groupby('cookie_id').pointer_y.transform('mean')
    points['cx2'] = points.cx**2
    points['cy2'] = points.cy**2
    points['cxcy'] = points.cx * points.cy
    points['radius'] = np.hypot(points.cx, points.cy)
    moments = points.groupby('cookie_id')[['cx2', 'cy2', 'cxcy']].mean()
    trace = moments.cx2 + moments.cy2
    rms_radius = np.sqrt(trace)

    features['xy_abs_correlation'] = (
        moments.cxcy.abs() / np.sqrt(moments.cx2 * moments.cy2).replace(0, np.nan)
    ).clip(0, 1).where(n_points.ge(3))
    # Формула для двух собственных значений ковариационной матрицы 2x2.
    features['xy_anisotropy'] = (
        np.sqrt((moments.cx2 - moments.cy2)**2 + 4 * moments.cxcy**2)
        / trace.where(trace.gt(0))
    ).clip(0, 1).where(n_points.ge(3))
    radii = points.groupby('cookie_id').radius
    features['xy_radius_cv'] = (
        radii.std(ddof=0) / radii.mean().replace(0, np.nan)
    ).where(n_points.ge(3))
    return features, rms_radius

## 8. Перемещения между наблюдаемыми точками

Координаты записаны в моменты событий: это **не непрерывная траектория мыши**.
Шаг допустим только между соседними web-событиями с полными XY, однозначным
временем и паузой `0 < dt <= 1800` секунд. Пропуск, другое устройство,
длинный перерыв или несколько событий в одну секунду разрывают последовательность.

Рассчитываются доля нулевых смещений, медиана длины смещения / RMS-радиус облака,
std/mean длины, доля повторных векторов и средний косинус между соседними ненулевыми
векторами. Значение косинуса +1 означает одинаковое направление, -1 — обратное.
Эти величины не интерпретируются как физическая скорость или длина реального пути.

В раннем обучении доли нулевых смещений и повторных векторов везде, где определены,
равны нулю: там они несут только информацию о доступности статистики. Прирост
широкого XY-набора не доказывает отдельную пользу каждого его компонента.

In [9]:
def pointer_motion_features(events, index, rms_radius):
    """Пять характеристик смещений с явными разрывами последовательности."""
    moved = events.copy()
    paired = moved.platform.eq('web') & np.isfinite(moved[['pointer_x', 'pointer_y']]).all(axis=1)
    unique_time = moved.groupby(['cookie_id', 'event_ts']).event_ts.transform('size').eq(1)
    point_ok = paired & unique_time
    previous_ok = point_ok.groupby(moved.cookie_id).shift().fillna(False).astype(bool)
    dt = moved.groupby('cookie_id').event_ts.diff().dt.total_seconds()

    # diff вычисляется до удаления пропусков: через неизвестную точку мостик не строится.
    moved['dx'] = moved.groupby('cookie_id').pointer_x.diff()
    moved['dy'] = moved.groupby('cookie_id').pointer_y.diff()
    step_ok = point_ok & previous_ok & dt.gt(0) & dt.le(1800)
    moved['step'] = np.hypot(moved.dx, moved.dy).where(step_ok)
    steps = moved.groupby('cookie_id').step
    n_steps = steps.count()
    features = pd.DataFrame(index=index)
    features['xy_zero_step_share'] = (
        moved.step.eq(0).astype(float).where(step_ok).groupby(moved.cookie_id).mean()
    )
    features['xy_step_median_normalized'] = steps.median() / rms_radius.replace(0, np.nan)
    features['xy_step_cv'] = (
        steps.std(ddof=0) / steps.mean().replace(0, np.nan)
    ).where(n_steps.ge(2))

    vectors = moved.loc[step_ok].groupby(['cookie_id', 'dx', 'dy']).size()
    features['xy_repeated_vector_share'] = (
        1 - vectors.groupby(level=0).size() / n_steps
    ).where(n_steps.ge(2))

    # Поворот существует только между двумя соседними допустимыми ненулевыми шагами.
    previous_dx = moved.groupby('cookie_id').dx.shift()
    previous_dy = moved.groupby('cookie_id').dy.shift()
    previous_step = moved.groupby('cookie_id').step.shift()
    turn_ok = moved.step.gt(0) & previous_step.gt(0)
    cosine = (
        (moved.dx * previous_dx + moved.dy * previous_dy) / (moved.step * previous_step)
    ).clip(-1, 1).where(turn_ok)
    features['xy_turn_cos_mean'] = cosine.groupby(moved.cookie_id).mean()
    return features

## 9. Одна строка признаков на cookie

Расчет не обучает преобразования на общей выборке: каждая статистика относится
только к одной cookie. Явный `reindex` и порядок `FEATURE_COLUMNS` предотвращают
перестановку строк и столбцов. Нулем заполняются только счетчики, прочие
неопределенные значения сохраняются как NaN и обрабатываются CatBoost.

In [10]:
def build_feature_matrix(events, meta):
    """Собрать только выбранные 42 признака в порядке метаданных."""
    index = pd.Index(meta.cookie_id, name='cookie_id')
    cloud, rms_radius = pointer_cloud_features(events, index)
    blocks = [
        count_features(events, index),
        rhythm_features(events, index),
        diversity_features(events, index),
        cloud,
        pointer_motion_features(events, index, rms_radius),
    ]
    features = pd.concat(blocks, axis=1).reindex(index)
    assert features.columns.is_unique and set(features.columns) == set(FEATURE_COLUMNS)
    features = features[FEATURE_COLUMNS].astype(float)
    assert not np.isinf(features.to_numpy()).any()
    return features


features = build_feature_matrix(events, meta)
X_train = features.loc[train.cookie_id]
X_test = features.loc[test.cookie_id]
assert X_train.index.tolist() == train.cookie_id.tolist()
assert X_test.index.tolist() == test.cookie_id.tolist()
assert X_train.shape[1] == X_test.shape[1] == 42
print('Матрицы признаков:', X_train.shape, X_test.shape)
display(X_train.head())

Матрицы признаков: (11091, 42) (4909, 42)


,gap_median,gap_min,gap_iqr_over_median,item_unique_share,category_entropy,location_unique_share,gap_le_10s_share,location_top_share,gap_unique_share,query_top_share,...,xy_repeat_share,xy_top_point_share,xy_abs_correlation,xy_anisotropy,xy_radius_cv,xy_zero_step_share,xy_step_median_normalized,xy_step_cv,xy_turn_cos_mean,xy_repeated_vector_share
cookie_id,,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,21.0,1.0,0.977273,1.000000,0.867563,0.714286,0.333333,0.428571,1.000000,0.333333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ck_7e4de46eeab82974,57.5,2.0,1.081197,0.593750,0.000000,0.324324,0.200000,0.324324,0.900000,0.250000,...,0.0,0.026316,0.265600,0.596178,0.524020,0.0,1.081543,0.553769,-0.310728,0.0
ck_9320229ef6304522,22.0,1.0,2.543478,0.950000,1.832968,0.272727,0.171429,0.666667,0.771429,0.230769,...,0.0,0.028571,0.045080,0.577787,0.371817,0.0,1.234444,0.455096,-0.718999,0.0
ck_30ccd25bc1714ed9,54.5,6.0,0.590090,0.555556,0.000000,0.200000,0.050000,0.500000,0.900000,0.333333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ck_a77c5f05948cdeef,52.0,22.0,1.396226,0.727273,1.278780,0.222222,0.000000,0.740741,0.962963,0.166667,...,0.0,0.037037,0.253377,0.460213,0.469067,0.0,1.132718,0.448870,-0.126723,0.0


## 10. Метрика и временная проверка

Ниже воспроизведена логика предоставленного `metric.py`: оценки сортируются
по убыванию, одинаковые score обрабатываются одной группой, берется максимум
precision среди всех допустимых порогов. Проверки на маленьких примерах защищают
от ошибочного разделения равных score и выбора только первого допустимого порога.

Фиксированная модель проверяется на трех временных отрезках. В каждом случае
обучение видит только более ранние даты. Последний отрезок 17–19 апреля уже
использовался в ходе работы; считать его новым независимым тестом нельзя.
Результаты этой ячейки не меняют ни параметры, ни список признаков.

In [11]:
def precision_at_recall(y_true, score, required_recall=0.70):
    """Официальная логика: максимум precision при recall не ниже заданного."""
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    if y_true.shape != score.shape:
        raise ValueError('Размеры меток и прогнозов не совпадают.')
    n_positive = int(y_true.sum())
    if n_positive == 0:
        return float('nan')

    order = np.argsort(-score, kind='mergesort')
    labels, sorted_score = y_true[order], score[order]
    true_positive = np.cumsum(labels)
    predicted_positive = np.arange(1, len(labels) + 1)
    # Порог нельзя поставить внутри группы одинаковых оценок.
    group_ends = np.r_[sorted_score[1:] != sorted_score[:-1], True]
    precision = true_positive[group_ends] / predicted_positive[group_ends]
    recall = true_positive[group_ends] / n_positive
    allowed = recall >= required_recall
    return float(precision[allowed].max()) if allowed.any() else 0.0


assert precision_at_recall([1, 0, 1, 1, 1], [5, 4, 3, 2, 1]) == 0.8
assert precision_at_recall([1, 1, 0, 0], [0.5] * 4) == 0.5
assert precision_at_recall([0, 0, 1, 1], [0.5] * 4) == 0.5

validation_rows = []
validation_periods = [
    ('development_1', '2026-04-12', '2026-04-15'),
    ('development_2', '2026-04-15', '2026-04-17'),
    ('final_comparison', '2026-04-17', '2026-04-20'),
]
for name, start, end in validation_periods:
    fit_mask = train.window_start_ts.lt(start).to_numpy()
    valid_mask = (train.window_start_ts.ge(start) & train.window_start_ts.lt(end)).to_numpy()
    validation_model = make_model()
    validation_model.fit(X_train.loc[fit_mask], y[fit_mask])
    score = validation_model.predict_proba(X_train.loc[valid_mask])[:, 1]
    precision, recall, _ = precision_recall_curve(y[valid_mask], score)
    validation_rows.append({
        'period': name, 'start': start, 'end_exclusive': end,
        'n_train': int(fit_mask.sum()), 'n_valid': int(valid_mask.sum()),
        'n_valid_bots': int(y[valid_mask].sum()),
        'P@R>=0.70': precision_at_recall(y[valid_mask], score),
        'PR-AUC': auc(recall, precision), 'ROC-AUC': roc_auc_score(y[valid_mask], score),
    })

validation_results = pd.DataFrame(validation_rows)
display(validation_results.round(5))

,period,start,end_exclusive,n_train,n_valid,n_valid_bots,P@R>=0.70,PR-AUC,ROC-AUC
0,development_1,2026-04-12,2026-04-15,5113,2486,190,0.50763,0.69500,0.91263
1,development_2,2026-04-15,2026-04-17,7599,1541,125,0.63309,0.73540,0.92479
2,final_comparison,2026-04-17,2026-04-20,9140,1951,160,0.55340,0.69705,0.90644


## 11. Интерпретация и ограничения

У варианта без XY на последнем отрезке Precision@Recall≥0.70 был 0.53081;
у XY-варианта — 0.55340. При 160 ботах оценка шумная: ориентировочный парный
bootstrap-интервал прибавки составляет [-0.052, +0.149] и включает ноль.
Это перспективный локальный результат, а не подтверждение улучшения скрытого теста.

Важности ниже относятся к модели, обученной до 17 апреля. Они помогают описать
модель, но не используются для нового отбора и не доказывают причинность признаков.
Будущие источники ботов и состав устройств могут отличаться от текущей выборки.

In [12]:
# После цикла validation_model — модель последнего временного сравнения.
importance = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance': validation_model.feature_importances_,
}).sort_values('importance', ascending=False)
display(importance.head(15).round(3))

,feature,importance
0,gap_median,10.857
1,gap_min,6.315
2,gap_iqr_over_median,6.189
3,item_unique_share,5.251
35,xy_anisotropy,4.236
4,category_entropy,3.625
14,event_entropy,3.548
5,location_unique_share,3.396
21,category_unique_share,3.063
13,gap_gt_30m_share,3.026


## 12. Обучение на всем train и сохранение submission.csv

Для test создается **новая** модель и обучается на всех размеченных cookie.
В файл идут вероятности положительного класса в исходном порядке `test.csv`.
Порог, `target`, индекс DataFrame и дополнительные столбцы не сохраняются.

Перед записью проверяются состав cookie, диапазон score и контрольная сумма CSV.
SHA-256 сравнивается с ранее сохраненным XY-результатом; сам старый файл не читается
и не копируется. При отличии выполнение завершится ошибкой до записи результата.
Это позволяет заметить изменение данных, кода, версии библиотеки или численного
окружения. Побайтовое совпадение проверено в указанном окружении.

In [13]:
final_model = make_model()
final_model.fit(X_train, y)
test_score = final_model.predict_proba(X_test)[:, 1]
assert final_model.get_all_params()['task_type'] == 'CPU'

submission = pd.DataFrame({'cookie_id': test.cookie_id, 'score': test_score})
assert list(submission.columns) == ['cookie_id', 'score']
assert len(submission) == len(test) and submission.cookie_id.is_unique
assert submission.cookie_id.equals(test.cookie_id)
assert np.isfinite(test_score).all() and submission.score.between(0, 1).all()

# Явные UTF-8 и LF делают текстовый формат независимым от настроек перевода строк.
payload = submission.to_csv(index=False, lineterminator='\n').encode('utf-8')
expected_sha256 = 'c4593595afc593a22c276bc1a5e12c4626389b4da62c10674ba21ba2402808a8'
actual_sha256 = hashlib.sha256(payload).hexdigest()
if actual_sha256 != expected_sha256:
    raise RuntimeError(f'Submission отличается от эталонного XY-файла: SHA-256={actual_sha256}')

submission_path = PROJECT_DIR / 'submission.csv'
submission_path.write_bytes(payload)
print('Сохранено:', submission_path)
print('Строк:', len(submission))
print('SHA-256:', actual_sha256)
print('Побайтовое совпадение с исходным XY-submission подтверждено.')
display(submission.head())

Сохранено: /Users/slvic/ANYA/avito_bot_detection/bot_detection_challenge/submission.csv
Строк: 4909
SHA-256: c4593595afc593a22c276bc1a5e12c4626389b4da62c10674ba21ba2402808a8
Побайтовое совпадение с исходным XY-submission подтверждено.


,cookie_id,score
0,ck_315fb710a0e371e7,0.000917
1,ck_a76ee3b3e3e522fd,0.295030
2,ck_94c9a4d382689e82,0.283997
3,ck_8eaf9509ad9462a0,0.010631
4,ck_9a88a5a989cb5bc6,0.019053
